# 08 — Synthetic Contact Generation

## 1. Introduction and objectives

This notebook (Phase 1E-e) generates the **synthetic contacts table** of CallVerse: one row per customer contact, linked to a real Olist order, with an intent, a timestamp and an example customer message from Bitext.

Contacts are drawn at random using **exactly** the assumptions of `configs/data.yaml`, already analysed in notebook 06. Each row is labelled `is_synthetic = True`.

**Technical safeguards**

| Risk | Safeguard |
|---|---|
| Different results on every run | Fixed random seed, and a test that two runs give identical tables |
| Drift between notebooks 06 and 08 | Eligible-order counts must match notebook 06's export exactly |
| Impossible timelines | Each timestamp is drawn inside the order's allowed interval, then every row is checked |
| Volumes not matching the assumptions | Statistical tests per situation and per intent |
| Uneven reuse of Bitext messages | Messages are cycled evenly within each intent; reuse is measured |
| Rare intents (imbalance) | Kept because realistic; zero-day shares are measured and intent groups are provided |
| Leakage | The cause of each contact (`generator_*` columns) is marked as generator-only |
| Edge effects | Contact volume at the start and end of the window is measured |

> **Synthetic data.** No row of this table was observed. Orders are real Olist orders; whether and when they led to a contact is simulated.

## 2. Imports, paths and inputs

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_FILE = PROJECT_ROOT / "configs" / "data.yaml"
BITEXT_FILE = PROJECT_ROOT / "data" / "raw" / "bitext" / "bitext-retail-ecommerce-llm-chatbot-training-dataset.csv"

required_files = [
    CONFIG_FILE, BITEXT_FILE,
    PROCESSED_DIR / "olist_orders_analytical.parquet",
    PROCESSED_DIR / "bitext_intent_catalog.csv",
    PROCESSED_DIR / "contact_expected_by_situation.csv",
    PROCESSED_DIR / "contact_hour_profile.csv",
    PROCESSED_DIR / "daily_calendar.parquet",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing inputs: {missing_files}. Run notebooks 03 to 07 first and check data/raw/bitext/.")

with open(CONFIG_FILE, encoding="utf-8") as config_file:
    contact_config = yaml.safe_load(config_file)["generation"]["contacts"]
situations_config = contact_config["situations"]

orders = pd.read_parquet(PROCESSED_DIR / "olist_orders_analytical.parquet")
intent_catalog = pd.read_csv(PROCESSED_DIR / "bitext_intent_catalog.csv")
expected_by_situation = pd.read_csv(PROCESSED_DIR / "contact_expected_by_situation.csv")
hour_profile = pd.read_csv(PROCESSED_DIR / "contact_hour_profile.csv")
daily_calendar = pd.read_parquet(PROCESSED_DIR / "daily_calendar.parquet")
bitext = pd.read_csv(BITEXT_FILE)

window_orders = orders[orders["in_analysis_window"]].copy()
EFFECTIVE_DATA_END = pd.Timestamp("2018-08-31")  # derived in notebook 04, section 10.2
LAST_CONTACT_MOMENT = EFFECTIVE_DATA_END + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
SEED = contact_config["random_seed"]

print("Config version:", contact_config["version"], "| seed:", SEED)
print("Orders in the analysis window:", len(window_orders))
print("Bitext messages:", len(bitext))

Config version: 1 | seed: 42
Orders in the analysis window: 99092
Bitext messages: 44884


## 3. Eligible orders per situation

The situation rules and the timing-feasibility rule are the same as in notebook 06 (sections 5 and 6). To guarantee the two notebooks stay consistent, the eligible-order counts are compared with notebook 06's export, and the notebook stops if they differ.

In [2]:
# Situation rules (identical to notebook 06, section 5)
delivered = window_orders["has_customer_delivery_timestamp"]
late = window_orders["delivered_after_estimated_date"].fillna(False)
seller_late = window_orders["carrier_after_shipping_limit"].fillna(False)
overdue = window_orders["overdue_without_delivery"]
canceled_or_unavailable = window_orders["is_canceled"] | window_orders["is_unavailable"]
low_review_with_comment = (window_orders["review_is_low_score"].fillna(False)
                           & window_orders["review_has_message"].fillna(False))
payment_anomaly = ((window_orders["payment_minus_order_value"].abs() > 1)
                   | (window_orders["primary_payment_type"] == "not_defined"))

situation_masks = {
    "overdue_never_delivered": overdue,
    "delivered_late": delivered & late,
    "seller_shipped_late": delivered & ~late & seller_late,
    "on_time_tracking_check": delivered & ~late & ~seller_late,
    "canceled_or_unavailable": canceled_or_unavailable,
    "post_delivery_problem": delivered & low_review_with_comment,
    "payment_anomaly": payment_anomaly.fillna(False),
    "change_before_shipping": ~canceled_or_unavailable,
    "invoice_request": window_orders["has_payments"],
}
if set(situation_masks) != set(situations_config):
    raise ValueError("Situations in the notebook and in configs/data.yaml must match.")

# Timing feasibility (identical to notebook 06, section 6)
feasible_masks = {}
for situation_name, mask in situation_masks.items():
    timing = situations_config[situation_name]["timing"]
    anchor_day = window_orders[timing["anchor"]].dt.normalize()
    earliest_day = anchor_day + pd.to_timedelta(timing["min_days"], unit="D")
    feasible = mask & anchor_day.notna() & (earliest_day <= EFFECTIVE_DATA_END)
    if "before" in timing:
        before_day = window_orders[timing["before"]].dt.normalize()
        feasible = feasible & before_day.notna() & (earliest_day <= before_day)
    feasible_masks[situation_name] = feasible

consistency = expected_by_situation[["situation", "feasible_orders"]].copy()
consistency["feasible_orders_here"] = consistency["situation"].map({name: int(mask.sum()) for name, mask in feasible_masks.items()})
consistency["match"] = consistency["feasible_orders"] == consistency["feasible_orders_here"]
display(consistency)
if not consistency["match"].all():
    raise ValueError("Eligible orders differ from notebook 06. Re-run notebook 06 or align the rules.")

,situation,feasible_orders,feasible_orders_here,match
0,overdue_never_delivered,1696,1696,True
1,delivered_late,6532,6532,True
2,seller_shipped_late,3986,3986,True
3,on_time_tracking_check,82897,82897,True
4,canceled_or_unavailable,1181,1181,True
5,post_delivery_problem,9169,9169,True
6,payment_anomaly,246,246,True
7,change_before_shipping,97304,97304,True
8,invoice_request,99092,99092,True


## 4. Message pools from Bitext

For each intent we keep the Bitext messages **without** the tags listed in `messages.exclude_tags` (offensive language `W`, as decided in Phase 1E-b). The table shows how many clean messages each intent has.

In [3]:
exclude_tags = contact_config["messages"]["exclude_tags"]
order_linked_intents = sorted(intent_catalog.loc[intent_catalog["proposed_scope"] == "order_linked", "intent"])

is_excluded = pd.Series(False, index=bitext.index)
for tag in exclude_tags:
    is_excluded = is_excluded | bitext["tags"].str.contains(tag, regex=False)
clean_bitext = bitext[~is_excluded & bitext["intent"].isin(order_linked_intents)]

message_pools = {}
pool_rows = []
for intent in order_linked_intents:
    pool_index = clean_bitext.index[clean_bitext["intent"] == intent].to_numpy()  # Bitext row numbers
    message_pools[intent] = pool_index
    pool_rows.append({"intent": intent, "all_messages": int((bitext["intent"] == intent).sum()),
                      "clean_messages": len(pool_index)})
pool_df = pd.DataFrame(pool_rows)
display(pool_df)
if (pool_df["clean_messages"] == 0).any():
    raise ValueError("An intent has no clean message to sample from.")

,intent,all_messages,clean_messages
0,cancel_order,996,423
1,change_order,961,310
2,damaged_delivery,992,420
3,delivery_issue,996,504
4,delivery_time,920,481
5,exchange_product,988,422
6,missing_item,721,358
7,payment_issue,996,487
8,product_issue,992,495
9,refund_status,999,506


## 5. The generator

For each situation the generator:

1. **Decides which eligible orders contact support**: one random draw per order with the situation's probability.
2. **Chooses the intent** with the situation's intent split.
3. **Chooses the contact day**: a whole number of days after the anchor, between `min_days` and an upper limit. The upper limit is the smallest of `max_days`, the "before" event and the effective data end, so the day is always allowed.
4. **Chooses the time of day** from the hourly profile (notebook 06, section 10).
5. **Enforces the timeline.** The contact must be after the anchor event and after the purchase, and before the "before" event and the data end. If the drawn time falls outside the allowed part of that day, a new time is drawn from the hourly profile **restricted to the allowed hours**, so the time-of-day profile is preserved. If no moment of that day is allowed (very rare, for example a delivery just after midnight), the contact is dropped and counted.

Then each contact receives a Bitext message for its intent. Messages are taken from a shuffled pool and **cycled**, so each message is reused as evenly as possible.

Everything is driven by one random generator created from the seed, so the same seed always gives the same table.

In [4]:
hour_values = hour_profile["hour"].to_numpy()
hour_shares = hour_profile["share"].to_numpy() / hour_profile["share"].sum()

def generate_contacts(seed):
    rng = np.random.default_rng(seed)
    situation_frames = []
    dropped_rows = []

    for situation_name, settings in situations_config.items():
        timing = settings["timing"]
        eligible = window_orders[feasible_masks[situation_name]]

        # 1. Which eligible orders contact support
        contacts_mask = rng.random(len(eligible)) < settings["probability"]
        chosen = eligible[contacts_mask]
        count = len(chosen)

        # 2. Intent
        intent_names = list(settings["intents"])
        intent_shares = np.array(list(settings["intents"].values()))
        intents = rng.choice(intent_names, size=count, p=intent_shares / intent_shares.sum())

        # 3. Contact day: offset between min_days and an upper limit that respects every constraint
        anchor_time = chosen[timing["anchor"]]
        anchor_day = anchor_time.dt.normalize()
        upper_days = np.full(count, timing["max_days"])
        upper_days = np.minimum(upper_days, (EFFECTIVE_DATA_END - anchor_day).dt.days.to_numpy())
        if "before" in timing:
            upper_days = np.minimum(upper_days, (chosen[timing["before"]].dt.normalize() - anchor_day).dt.days.to_numpy())
        offsets = rng.integers(timing["min_days"], upper_days + 1)
        contact_day = anchor_day + pd.to_timedelta(offsets, unit="D")

        # 4. Time of day from the hourly profile
        hours = rng.choice(hour_values, size=count, p=hour_shares)
        seconds = rng.integers(0, 3600, size=count)
        contact_time = contact_day + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(seconds, unit="s")

        # 5. Allowed interval for that day
        # x.where(condition, y) keeps x where the condition holds, otherwise takes y
        purchase_time = chosen["order_purchase_timestamp"]
        lower_bound = anchor_time.where(anchor_time >= purchase_time, purchase_time)
        upper_bound = pd.Series(LAST_CONTACT_MOMENT, index=chosen.index)
        if "before" in timing:
            before_time = chosen[timing["before"]]
            upper_bound = before_time.where(before_time < upper_bound, upper_bound)
        day_last_second = contact_day + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        day_start = contact_day.where(contact_day >= lower_bound, lower_bound)
        day_end = day_last_second.where(day_last_second <= upper_bound, upper_bound)

        possible = day_start <= day_end
        outside = (contact_time < day_start) | (contact_time > day_end)

        # Times outside the allowed part of the day are redrawn from the hourly profile,
        # restricted to the allowed hours, so the time-of-day profile is preserved
        contact_time = contact_time.copy()
        for position in np.flatnonzero((outside & possible).to_numpy()):
            start = day_start.iloc[position]
            end = day_end.iloc[position]
            allowed_hours = np.arange(start.hour, end.hour + 1)
            weights = hour_shares[allowed_hours] / hour_shares[allowed_hours].sum()
            hour = rng.choice(allowed_hours, p=weights)
            hour_begin = contact_day.iloc[position] + pd.Timedelta(hours=int(hour))
            slot_start = max(start, hour_begin)
            slot_end = min(end, hour_begin + pd.Timedelta(seconds=3599))
            contact_time.iloc[position] = slot_start + (slot_end - slot_start) * rng.random()
        contact_time = contact_time.dt.floor("s")

        frame = pd.DataFrame({
            "order_id": chosen["order_id"],
            "customer_unique_id": chosen["customer_unique_id"],
            "contact_timestamp": contact_time,
            "intent": intents,
            "generator_situation": situation_name,
            "generator_anchor_event": timing["anchor"],
            "generator_days_after_anchor": offsets,
            "time_adjusted_to_timeline": outside & possible,
        })
        dropped_rows.append({"situation": situation_name, "dropped_no_allowed_time": int((~possible).sum())})
        situation_frames.append(frame[possible.to_numpy()])

    contacts = pd.concat(situation_frames, ignore_index=True)
    contacts = contacts.sort_values(["contact_timestamp", "order_id", "intent"]).reset_index(drop=True)

    # Messages: shuffled pool per intent, cycled so reuse is as even as possible
    bitext_rows = np.empty(len(contacts), dtype=int)
    for intent in order_linked_intents:
        positions = np.flatnonzero(contacts["intent"].to_numpy() == intent)
        shuffled_pool = rng.permutation(message_pools[intent])
        bitext_rows[positions] = shuffled_pool[np.arange(len(positions)) % len(shuffled_pool)]
    contacts["bitext_row_id"] = bitext_rows

    contacts.insert(0, "contact_id", [f"SC{number:06d}" for number in range(1, len(contacts) + 1)])
    return contacts, pd.DataFrame(dropped_rows)

contacts, dropped = generate_contacts(SEED)
print("Generated contacts:", len(contacts))
display(dropped)

Generated contacts: 9179


,situation,dropped_no_allowed_time
0,overdue_never_delivered,0
1,delivered_late,0
2,seller_shipped_late,0
3,on_time_tracking_check,1
4,canceled_or_unavailable,0
5,post_delivery_problem,0
6,payment_anomaly,0
7,change_before_shipping,1
8,invoice_request,0


### 5.1 Reproducibility

Running the generator a second time with the same seed must give an identical table; a different seed must give a different one.

In [5]:
contacts_again, _ = generate_contacts(SEED)
contacts_other_seed, _ = generate_contacts(SEED + 1)
print("Same seed gives an identical table:", contacts.equals(contacts_again))
print("Different seed gives a different table:", not contacts.equals(contacts_other_seed))
if not contacts.equals(contacts_again):
    raise ValueError("The generator is not reproducible.")

Same seed gives an identical table: True
Different seed gives a different table: True


## 6. Complete the contacts table

We add the message text and tags from Bitext, the intent category and group, calendar fields of the contact day (known in advance), and the synthetic labels.

In [6]:
intent_group_by_intent = {}
for group_name, group_intents in contact_config["intent_groups"].items():
    for intent in group_intents:
        intent_group_by_intent[intent] = group_name
if set(intent_group_by_intent) != set(order_linked_intents):
    raise ValueError("Every order-linked intent must belong to exactly one intent group.")

contacts["message_text"] = contacts["bitext_row_id"].map(bitext["instruction"])
contacts["bitext_intent"] = contacts["bitext_row_id"].map(bitext["intent"])
contacts["bitext_tags"] = contacts["bitext_row_id"].map(bitext["tags"])
contacts["intent_category"] = contacts["intent"].map(intent_catalog.set_index("intent")["category"])
contacts["intent_group"] = contacts["intent"].map(intent_group_by_intent)

contacts["contact_date"] = contacts["contact_timestamp"].dt.normalize()
contacts["contact_hour"] = contacts["contact_timestamp"].dt.hour
contacts["contact_day_of_week"] = contacts["contact_timestamp"].dt.day_name()
calendar_by_date = daily_calendar.set_index("date")
for column in ["in_promo_window", "promo_group", "is_public_holiday"]:
    contacts[column] = contacts["contact_date"].map(calendar_by_date[column])

contacts["is_synthetic"] = True
contacts["generator_version"] = contact_config["version"]
contacts["generator_seed"] = SEED
contacts["scenario"] = "base"

column_order = [
    "contact_id", "contact_timestamp", "contact_date", "contact_hour", "contact_day_of_week",
    "intent", "intent_category", "intent_group", "message_text", "bitext_row_id", "bitext_tags",
    "order_id", "customer_unique_id",
    "in_promo_window", "promo_group", "is_public_holiday",
    "generator_situation", "generator_anchor_event", "generator_days_after_anchor", "time_adjusted_to_timeline",
    "is_synthetic", "generator_version", "generator_seed", "scenario",
]
contacts = contacts[column_order]
print("Contacts table:", contacts.shape)
display(contacts.head(5))

Contacts table: (9179, 24)


,contact_id,contact_timestamp,contact_date,contact_hour,contact_day_of_week,intent,intent_category,intent_group,message_text,bitext_row_id,...,promo_group,is_public_holiday,generator_situation,generator_anchor_event,generator_days_after_anchor,time_adjusted_to_timeline,is_synthetic,generator_version,generator_seed,scenario
0,SC000001,2017-01-11 14:26:47,2017-01-11,14,Wednesday,delivery_time,DELIVERY,wismo,wanna know more about the delivery date give m...,11192,...,<NA>,False,seller_shipped_late,latest_shipping_limit_date,2,True,True,1,42,base
1,SC000002,2017-01-13 17:44:20,2017-01-13,17,Friday,change_order,ORDER,order_admin,i purchased a product but i chose a wrong mode...,6596,...,<NA>,False,change_before_shipping,order_purchase_timestamp,1,False,True,1,42,base
2,SC000003,2017-01-14 00:47:28,2017-01-14,0,Saturday,delivery_time,DELIVERY,wismo,i have to see the delivery time how could i do it,10880,...,<NA>,False,seller_shipped_late,latest_shipping_limit_date,1,True,True,1,42,base
3,SC000004,2017-01-15 19:05:38,2017-01-15,19,Sunday,payment_issue,PAYMENT,order_admin,"I would like to solve payment issues , can I g...",18797,...,<NA>,False,payment_anomaly,order_purchase_timestamp,1,False,True,1,42,base
4,SC000005,2017-01-17 12:58:33,2017-01-17,12,Tuesday,request_invoice,ORDER,order_admin,i made a purchaxe and i didnt get an invoice r...,26169,...,<NA>,False,invoice_request,order_purchase_timestamp,5,False,True,1,42,base


**Column groups**

- **Contact**: `contact_id`, time fields, `intent`, its category and group, `message_text`.
- **Link to Olist**: `order_id` and `customer_unique_id` (join with the order table for context).
- **Calendar**: promotion and holiday fields of the contact day.
- **Generator-only** (`generator_*`, `time_adjusted_to_timeline`): *why* and *how* the contact was generated. They are useful to evaluate models (for example "did the twin route late-delivery contacts well?"), but **must not be used as inputs** to forecasting or agents. `generator_situation` is, by construction, the hidden cause of the contact.
- **Labels**: `is_synthetic`, `generator_version`, `generator_seed`, `scenario`.

## 7. Validation

### 7.1 Volumes match the assumptions

For each situation the number of contacts is random, following a binomial distribution with *n* = eligible orders and *p* = probability. We compute a **z-score**: how many standard deviations the generated count is from the expected count. Values between −4 and 4 are normal random variation.

We use 4 rather than the usual 2 or 3 because we run many tests: with dozens of tests at a threshold of 2, a few would fail by pure chance even with a correct generator.

In [7]:
Z_LIMIT = 4

volume_rows = []
for situation_name, settings in situations_config.items():
    eligible_count = int(feasible_masks[situation_name].sum())
    probability = settings["probability"]
    expected = eligible_count * probability
    standard_deviation = np.sqrt(eligible_count * probability * (1 - probability))
    generated = int((contacts["generator_situation"] == situation_name).sum())
    dropped_count = int(dropped.loc[dropped["situation"] == situation_name, "dropped_no_allowed_time"].sum())
    z_score = (generated + dropped_count - expected) / standard_deviation if standard_deviation > 0 else 0.0
    volume_rows.append({"situation": situation_name, "eligible_orders": eligible_count, "expected": round(expected, 1),
                        "generated": generated, "dropped": dropped_count, "z_score": round(z_score, 2),
                        "within_limit": abs(z_score) <= Z_LIMIT})
volume_check = pd.DataFrame(volume_rows)
display(volume_check)
print("Total generated:", int(volume_check["generated"].sum()), "| total expected:", round(volume_check["expected"].sum()))

,situation,eligible_orders,expected,generated,dropped,z_score,within_limit
0,overdue_never_delivered,1696,1017.6,1021,0,0.17,True
1,delivered_late,6532,2286.2,2275,0,-0.29,True
2,seller_shipped_late,3986,199.3,198,0,-0.09,True
3,on_time_tracking_check,82897,1243.5,1279,1,1.04,True
4,canceled_or_unavailable,1181,590.5,597,0,0.38,True
5,post_delivery_problem,9169,2750.7,2718,0,-0.75,True
6,payment_anomaly,246,73.8,84,0,1.42,True
7,change_before_shipping,97304,486.5,504,1,0.84,True
8,invoice_request,99092,495.5,503,0,0.34,True


Total generated: 9179 | total expected: 9144


In [8]:
# Intent split inside each situation
split_rows = []
for situation_name, settings in situations_config.items():
    situation_contacts = contacts[contacts["generator_situation"] == situation_name]
    situation_total = len(situation_contacts)
    for intent, share in settings["intents"].items():
        generated = int((situation_contacts["intent"] == intent).sum())
        expected = situation_total * share
        standard_deviation = np.sqrt(situation_total * share * (1 - share))
        z_score = (generated - expected) / standard_deviation if standard_deviation > 0 else 0.0
        split_rows.append({"situation": situation_name, "intent": intent, "share_config": share,
                           "share_generated": round(generated / situation_total, 3) if situation_total else None,
                           "z_score": round(z_score, 2), "within_limit": abs(z_score) <= Z_LIMIT})
split_check = pd.DataFrame(split_rows)
print("Intent-split tests:", len(split_check), "| outside the limit:", int((~split_check["within_limit"]).sum()))
display(split_check[split_check["situation"].isin(["delivered_late", "post_delivery_problem"])])

Intent-split tests: 28 | outside the limit: 0


,situation,intent,share_config,share_generated,z_score,within_limit
4,delivered_late,track_order,0.35,0.368,1.79,True
5,delivered_late,track_delivery,0.25,0.244,-0.67,True
6,delivered_late,delivery_time,0.20,0.209,1.05,True
7,delivered_late,delivery_issue,0.20,0.179,-2.46,True
18,post_delivery_problem,product_issue,0.20,0.202,0.31,True
19,post_delivery_problem,wrong_item,0.15,0.143,-1.00,True
20,post_delivery_problem,damaged_delivery,0.15,0.147,-0.41,True
21,post_delivery_problem,exchange_product,0.15,0.149,-0.15,True
22,post_delivery_problem,return_product,0.15,0.166,2.33,True
23,post_delivery_problem,missing_item,0.10,0.094,-1.07,True


### 7.2 Timeline, identifiers and messages

In [9]:
order_lookup = orders.set_index("order_id")
contact_orders = order_lookup.loc[contacts["order_id"]].reset_index()

purchase_time = contact_orders["order_purchase_timestamp"]
delivery_time = contact_orders["order_delivered_customer_date"]
carrier_time = contact_orders["order_delivered_carrier_date"]
estimate_day = contact_orders["order_estimated_delivery_date"]
contact_time = contacts["contact_timestamp"]
situation = contacts["generator_situation"]

wismo_before_delivery = situation.isin(["delivered_late", "on_time_tracking_check"])
before_handoff = situation.isin(["seller_shipped_late", "change_before_shipping"])
after_delivery = situation == "post_delivery_problem"
after_promise = situation.isin(["overdue_never_delivered", "delivered_late"])

timeline_checks = [
    ("Every contact is after the order's purchase", bool((contact_time >= purchase_time).all())),
    ("Every contact is on or before the effective data end", bool((contact_time <= LAST_CONTACT_MOMENT).all())),
    ("Late / in-transit tracking contacts happen before delivery",
     bool((contact_time[wismo_before_delivery] <= delivery_time[wismo_before_delivery]).all())),
    ("Seller-late and change-order contacts happen before carrier handoff",
     bool((contact_time[before_handoff] <= carrier_time[before_handoff]).all())),
    ("Post-delivery problem contacts happen after delivery",
     bool((contact_time[after_delivery] >= delivery_time[after_delivery]).all())),
    ("Late / overdue contacts happen after the promised date",
     bool((contact_time[after_promise] >= estimate_day[after_promise] + pd.Timedelta(days=1)).all())),
    ("Every contacted order is in the analysis window", bool(contact_orders["in_analysis_window"].all())),
    ("contact_id is unique", contacts["contact_id"].is_unique),
    ("At most one contact per order and situation", not contacts.duplicated(["order_id", "generator_situation"]).any()),
    ("Message intent (looked up in Bitext) equals contact intent",
     bool((contacts["bitext_row_id"].map(bitext["intent"]) == contacts["intent"]).all())),
    ("No excluded (offensive) message used",
     not contacts["bitext_tags"].apply(lambda tags: any(tag in tags for tag in exclude_tags)).any()),
    ("Every intent is an order-linked intent", bool(contacts["intent"].isin(order_linked_intents).all())),
    ("All contacts are labelled synthetic", bool(contacts["is_synthetic"].all())),
    ("Situation volumes within the z-score limit", bool(volume_check["within_limit"].all())),
    ("Intent splits within the z-score limit", bool(split_check["within_limit"].all())),
]
timeline_df = pd.DataFrame(timeline_checks, columns=["check", "passed"])
display(timeline_df)
if not timeline_df["passed"].all():
    raise ValueError("A contact validation check failed; see the table above.")
print("Contacts whose time was adjusted to fit the timeline:", int(contacts["time_adjusted_to_timeline"].sum()),
      f"({contacts['time_adjusted_to_timeline'].mean() * 100:.1f}%)")

,check,passed
0,Every contact is after the order's purchase,True
1,Every contact is on or before the effective da...,True
2,Late / in-transit tracking contacts happen bef...,True
3,Seller-late and change-order contacts happen b...,True
4,Post-delivery problem contacts happen after de...,True
5,Late / overdue contacts happen after the promi...,True
6,Every contacted order is in the analysis window,True
7,contact_id is unique,True
8,At most one contact per order and situation,True
9,Message intent (looked up in Bitext) equals co...,True


Contacts whose time was adjusted to fit the timeline: 857 (9.3%)


### 7.3 Time of day and message reuse

In [10]:
generated_hours = contacts["contact_hour"].value_counts(normalize=True).reindex(hour_values, fill_value=0)
hour_comparison = pd.DataFrame({"profile_percent": hour_shares * 100, "generated_percent": generated_hours.to_numpy() * 100},
                               index=hour_values).round(2)
max_hour_gap = (hour_comparison["generated_percent"] - hour_comparison["profile_percent"]).abs().max()
display(hour_comparison.T)
print(f"Largest gap between generated and profile hour shares: {max_hour_gap:.2f} percentage points")

reuse = contacts.groupby("intent")["bitext_row_id"].agg(contacts="size", distinct_messages="nunique", max_uses_of_one_message=lambda ids: ids.value_counts().max())
reuse["clean_pool"] = reuse.index.map(pool_df.set_index("intent")["clean_messages"])
reuse["average_uses"] = (reuse["contacts"] / reuse["distinct_messages"]).round(2)
display(reuse.sort_values("contacts", ascending=False))

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
profile_percent,2.41,1.18,0.51,0.27,0.20,0.19,0.51,1.24,2.98,4.81,...,6.61,6.48,6.72,6.19,5.81,6.03,6.23,6.26,5.83,4.15
generated_percent,3.15,1.31,0.54,0.26,0.27,0.15,0.36,1.19,2.78,4.85,...,6.19,6.44,6.67,6.10,5.15,5.90,6.47,6.00,6.55,4.80


Largest gap between generated and profile hour shares: 0.74 percentage points


,contacts,distinct_messages,max_uses_of_one_message,clean_pool,average_uses
intent,,,,,
track_order,1888,507,4,507,3.72
track_delivery,1320,527,3,527,2.50
delivery_time,912,481,2,481,1.90
delivery_issue,653,504,2,504,1.30
product_issue,550,495,2,495,1.11
change_order,504,310,2,310,1.63
request_invoice,503,495,2,495,1.02
return_product,451,451,1,625,1.00
exchange_product,405,405,1,422,1.00


The hour profile is matched closely. Remaining small gaps come from contacts whose allowed hours are limited by the timeline (for example, a "where is my order?" contact on the delivery day must happen before the delivery time), and from random variation.

With cycling, the most-used message of an intent is used at most once more than the least-used one. When an intent has fewer contacts than clean messages, no message is repeated at all.

## 8. Daily contact series and imbalance

The forecasting phase needs **daily counts**. We build a complete grid of every day of the analysis window × every intent, so days without contacts appear as 0 rather than being missing. The grid stops at the effective data end.

In [11]:
window_days = pd.date_range("2017-01-01", EFFECTIVE_DATA_END, freq="D")
grid = pd.MultiIndex.from_product([window_days, order_linked_intents], names=["date", "intent"])
counts = contacts.groupby(["contact_date", "intent"]).size()
counts.index = counts.index.set_names(["date", "intent"])
daily_by_intent = counts.reindex(grid, fill_value=0).rename("contacts").reset_index()
daily_by_intent["intent_group"] = daily_by_intent["intent"].map(intent_group_by_intent)

daily_by_group = daily_by_intent.groupby(["date", "intent_group"], as_index=False)["contacts"].sum()
daily_total = daily_by_intent.groupby("date", as_index=False)["contacts"].sum()

print("Daily-by-intent rows:", len(daily_by_intent), "=", len(window_days), "days x", len(order_linked_intents), "intents")
print("Contacts preserved in the grid:", int(daily_by_intent["contacts"].sum()) == len(contacts))

Daily-by-intent rows: 10336 = 608 days x 17 intents
Contacts preserved in the grid: True


In [12]:
def zero_day_summary(table, key):
    summary = table.groupby(key)["contacts"].agg(
        total="sum", mean_per_day="mean", max_per_day="max", zero_days=lambda values: int((values == 0).sum()))
    summary["zero_day_percent"] = (summary["zero_days"] / len(window_days) * 100).round(1)
    return summary.sort_values("total", ascending=False).round(2)

print("Per intent:")
display(zero_day_summary(daily_by_intent, "intent"))
print("Per intent group:")
display(zero_day_summary(daily_by_group, "intent_group"))
print("All contacts: zero days =", int((daily_total["contacts"] == 0).sum()),
      "| mean per day =", round(daily_total["contacts"].mean(), 1))

Per intent:


,total,mean_per_day,max_per_day,zero_days,zero_day_percent
intent,,,,,
track_order,1888,3.11,20,92,15.1
track_delivery,1320,2.17,12,144,23.7
delivery_time,912,1.50,11,213,35.0
delivery_issue,653,1.07,8,278,45.7
product_issue,550,0.90,7,279,45.9
change_order,504,0.83,6,278,45.7
request_invoice,503,0.83,5,282,46.4
return_product,451,0.74,5,313,51.5
exchange_product,405,0.67,7,339,55.8


Per intent group:


,total,mean_per_day,max_per_day,zero_days,zero_day_percent
intent_group,,,,,
wismo,4773,7.85,44,36,5.9
post_delivery,2718,4.47,19,69,11.3
order_admin,1091,1.79,8,127,20.9
cancel_refund,597,0.98,6,264,43.4


All contacts: zero days = 16 | mean per day = 15.1


**Reading the imbalance**

- Intents are naturally unbalanced: "where is my order?" intents dominate and some intents are rare. This is **realistic**, not an error, and it must not be "fixed" by rebalancing. The twin must face the real mix.
- What matters for forecasting is the share of **zero days**. A series that is zero on most days cannot be forecast well on its own. The intent groups have far fewer zero days than single intents, so the forecasting phase should forecast **groups** (or the total) and split them into intents with their average shares.
- For the Advisor and Quality agents, the rare intents can still be tested with targeted scenarios.

## 9. Edge effects

Contacts happen *after* order events, often days or weeks after the purchase. Two edge effects follow:

- **Start of the window:** contacts begin slowly, because the first contacts need orders placed before them. Orders before 2017 are outside the window, but Olist had very few, so this effect is small and short.
- **End of the window:** contacts that would fall after 31 August 2018 are missing (right censoring). At the same time purchases collapse in the last days of August (notebook 04). Contacts keep coming from orders placed weeks earlier, so **contacts per 100 orders rise sharply** at the very end. That rise is an artefact, not a real increase in contact rate. The last weekly row is also a partial week, because the grid stops on Friday 31 August.

In [13]:
weekly = daily_total.set_index("date")["contacts"].resample("W-SUN").sum()
weekly_orders = orders[orders["in_analysis_window"]].set_index("purchase_date").resample("W-SUN").size()
weekly_table = pd.DataFrame({"contacts": weekly, "orders_purchased": weekly_orders})
weekly_table["contacts_per_100_orders"] = (weekly_table["contacts"] / weekly_table["orders_purchased"] * 100).round(1)
print("First 4 weeks:")
display(weekly_table.head(4))
print("Last 8 weeks:")
display(weekly_table.tail(8))

First 4 weeks:


,contacts,orders_purchased,contacts_per_100_orders
2017-01-01,0,NaN,NaN
2017-01-08,0,46.0,0.0
2017-01-15,4,80.0,5.0
2017-01-22,7,197.0,3.6


Last 8 weeks:


,contacts,orders_purchased,contacts_per_100_orders
2018-07-15,55,1001.0,5.5
2018-07-22,74,1678.0,4.4
2018-07-29,106,1646.0,6.4
2018-08-05,123,2058.0,6.0
2018-08-12,198,1988.0,10.0
2018-08-19,197,1875.0,10.5
2018-08-26,157,1071.0,14.7
2018-09-02,105,130.0,80.8


**Recommendation:** the forecasting phase should end its training and test periods a few weeks before 31 August 2018 (for example mid-August), or treat the final weeks as incomplete. Ratios such as contacts per order should not be computed on the last weeks.

## 10. Export

In [14]:
contacts_path = PROCESSED_DIR / "synthetic_contacts.parquet"
daily_intent_path = PROCESSED_DIR / "synthetic_contacts_daily_by_intent.parquet"
daily_group_path = PROCESSED_DIR / "synthetic_contacts_daily_by_group.parquet"

contacts.to_parquet(contacts_path, index=False)
daily_by_intent.to_parquet(daily_intent_path, index=False)
daily_by_group.to_parquet(daily_group_path, index=False)

export_rows = []
for path, table in [(contacts_path, contacts), (daily_intent_path, daily_by_intent), (daily_group_path, daily_by_group)]:
    reloaded = pd.read_parquet(path)
    export_rows.append({"file": path.name, "rows": len(reloaded), "columns": len(reloaded.columns),
                        "identical_after_reload": reloaded.equals(table),
                        "size_mb": round(path.stat().st_size / 1024**2, 2)})
display(pd.DataFrame(export_rows))

,file,rows,columns,identical_after_reload,size_mb
0,synthetic_contacts.parquet,9179,24,True,0.96
1,synthetic_contacts_daily_by_intent.parquet,10336,4,True,0.01
2,synthetic_contacts_daily_by_group.parquet,2432,3,True,0.01


## 11. Summary and limitations

**Produced** (in `data/processed/`)

- `synthetic_contacts.parquet`: one row per synthetic contact, linked to an Olist order, with intent, timestamp, Bitext message, calendar fields and generator labels.
- `synthetic_contacts_daily_by_intent.parquet`: complete day × intent grid of contact counts (zeros included).
- `synthetic_contacts_daily_by_group.parquet`: the same per intent group, recommended for forecasting.

**Guarantees checked in this notebook**

- Reproducible: same seed, same table.
- Consistent with notebook 06: same eligible orders; volumes and intent splits within random variation.
- Coherent timelines: no contact before purchase, after the data end, or in contradiction with delivery and handoff times.
- Clean messages only, matching the contact's intent, reused evenly.
- One contact per order and situation.

**Limitations**

- **Synthetic.** Volumes, timing and intents follow the assumptions of `configs/data.yaml`. They are plausible (benchmark check in notebook 06) but not observed.
- **Generator columns are hidden causes.** Do not use `generator_*` columns as model inputs.
- **Right censoring** in the last weeks of August 2018 (section 9).
- **No channels, repeat contacts or handling times yet.** These belong to the simulation phase (queues, agents) or a later generator version.
- **English Bitext wording** for Brazilian orders (Phase 1E-b decision).